In [ ]:
#install langchain, aws, faiss library

!pip install -q langchain langchain-community langchain-aws faiss-cpu boto3 tiktoken

In [ ]:
#AWS Credentials Setup
import os

os.environ["AWS_ACCESS_KEY_ID"] = ""
os.environ["AWS_SECRET_ACCESS_KEY"] = ""
os.environ["AWS_DEFAULT_REGION"] = ""

In [ ]:
#Initialize Bedrock Client
import boto3

bedrock = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1"
)

In [ ]:
#Titan Embeddings Setup - Text is converted into vectors
from langchain_aws import BedrockEmbeddings

embeddings = BedrockEmbeddings(
    model_id="amazon.titan-embed-text-v1",
    client=bedrock
)

In [ ]:
#Load & Prepare Documents - Large text is split into smaller chunks

from langchain.text_splitter import RecursiveCharacterTextSplitter

documents = [
    "RAG stands for Retrieval Augmented Generation.",
    "FAISS is a vector database developed by Meta.",
    "Claude Sonnet is a powerful LLM by Anthropic."
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

docs = text_splitter.create_documents(documents)

In [ ]:
#Create FAISS Vector Store - Each chunk → converted into embedding and store in faiss vector db

from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)

In [ ]:
#Create Retriever - Retriever is created from FAISS

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [ ]:
#Claude Sonnet Model Setup - Reads context and Generates human like responses

from langchain_aws import ChatBedrock

llm = ChatBedrock(
    model_id="anthropic.claude-3-sonnet-20240229-v1:0",
    client=bedrock,
    model_kwargs={
        "temperature": 0.5,
        "max_tokens": 512
    }
)

In [ ]:
#Build RAG Chain - Retriever + LLM

from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

In [ ]:
#Query the RAG System
query = "What is FAISS?"

response = qa_chain.invoke({"query": query})

print("Answer:\n", response["result"])
print("\nSources:\n", response["source_documents"])